# Тема 7. Шумоподібний сигнал мовою Python

Ви відкрили Jupyter Notebook: файл, у якому пояснення чергуються з короткими програмами мовою Python. Програми цього файлу ховають коротке повідомлення під шумом і виділяють повідомлення за кодом. Так працює шумоподібний сигнал із рис. 7.3, в теми 7.

**Як працювати.** Jupyter Notebook складається з комірок: одні комірки містять текст, як ця, а інші програму. Клацніть першу комірку з програмою (вона розташована одразу під цим текстом) і натисніть Shift+Enter: комірка виконається, а під нею з’явиться результат. Так само виконуйте решту комірок по черзі, згори донизу.

In [ ]:
# Повідомлення і код. Кожен знак повідомлення програма записує шістьма бітами.
import matplotlib.pyplot as plt
import numpy as np

ALPHABET = "АБВГҐДЕЄЖЗИІЇЙКЛМНОПРСТУФХЦЧШЩЬЮЯ 0123456789.,-?"
ALPHABET += "*" * (64 - len(ALPHABET))    # зірочка: число, якому не відповідає жоден знак
MESSAGE = "РАДІО 262"

CHIPS = 100       # на скільки частин передавач ділить кожен біт
CODE = 7          # номер коду, який знають передавач і свій приймач
OTHER = 11        # номер коду, який пробує чужий приймач
NOISE = 2.0       # у скільки разів шум у середньому більший за сигнал


def to_bits(text):
    """Текст як послідовність бітів: шість бітів на знак."""
    return np.array([int(b) for sign in text for b in format(ALPHABET.index(sign), "06b")])


def to_text(bits):
    """Текст, який записано послідовністю бітів."""
    return "".join(ALPHABET[int("".join(str(b) for b in bits[i:i + 6]), 2)] for i in range(0, len(bits), 6))


def make_code(number):
    """Код: послідовність із CHIPS чисел +1 і -1, яку визначає номер коду."""
    return np.random.default_rng(number).integers(0, 2, CHIPS) * 2 - 1


bits = to_bits(MESSAGE)
code = make_code(CODE)
print(f"Повідомлення «{MESSAGE}»: {len(bits)} бітів")
print("Початок коду:", " ".join(f"{c:+d}" for c in code[:12]))

## Крок 1. Передавач і ефір

Передавач ділить кожен біт на `CHIPS` коротких частин. Біт 1 передавач надсилає як код, тобто як послідовність чисел +1 і −1. Біт 0 передавач надсилає як той самий код із протилежними знаками.

В ефірі до сигналу додається шум. Число `NOISE` задає, у скільки разів шум у середньому більший за сигнал. Потужність зростає як квадрат цього числа: за `NOISE = 2.0` шум у 4 рази потужніший за сигнал.

Наступна програма малює перші три біти: сигнал передавача і те, що приходить на антену.

In [ ]:
# Передавач: кожен біт стає кодом (біт 1) або кодом із протилежними знаками (біт 0).
signal = np.repeat(2 * bits - 1, CHIPS) * np.tile(code, len(bits))
# Ефір: до сигналу додається шум.
air = signal + NOISE * np.random.default_rng(20261117).standard_normal(len(signal))

shown = 3 * CHIPS
figure, (clean, noisy) = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
clean.step(range(shown), signal[:shown], where="post", color="tab:blue")
clean.set_title("Сигнал передавача без шуму: перші три біти")
noisy.plot(range(shown), air[:shown], color="tab:gray", linewidth=0.8)
noisy.set_title("Те саме в ефірі: сигнал разом із шумом")
noisy.set_xlabel("номер частини")
for picture in (clean, noisy):
    picture.grid(alpha=0.3)
    for edge in range(0, shown + 1, CHIPS):          # межі бітів
        picture.axvline(edge, color="black", linewidth=0.6, linestyle=":")
plt.tight_layout()
plt.show()

## Крок 2. Приймач

Приймач множить прийнятий сигнал на код, частину за частиною, і складає добутки кожного біта. Частини сигналу мають ті самі знаки, що й код, тому їхні добутки додаються один до одного. Шум має випадкові знаки, тому його добутки здебільшого знищують один одного.

Якщо сума більша за нуль, приймач читає біт 1, інакше біт 0. Чужий приймач не знає номера коду й пробує інший.

In [ ]:
def receive(number):
    """Приймач із кодом number: біти, які він прочитав."""
    sums = (air * np.tile(make_code(number), len(bits))).reshape(len(bits), CHIPS).sum(axis=1)
    return (sums > 0).astype(int)


for name, number in (("Свій код", CODE), ("Чужий код", OTHER)):
    read = receive(number)
    print(f"{name}: «{to_text(read)}», помилок: {(read != bits).sum()} із {len(bits)}")

## Завдання

Після кожної зміни виконуйте програми знову, згори донизу.

1. Запишіть, скільки бітів прочитав неправильно свій приймач і скільки чужий. Скільки помилок дало б угадування бітів навмання? Тепер дайте чужому приймачеві код передавача: замініть рядок `OTHER = 11` на `OTHER = 7`. Що він прочитав? Поверніть `OTHER = 11`.
2. Посильте шум: замініть рядок `NOISE = 2.0` на `NOISE = 5.0`, потім на `NOISE = 10.0`. Що сталося з повідомленням свого приймача?
3. Залиште `NOISE = 10.0` і подовжте код: замініть рядок `CHIPS = 100` на `CHIPS = 400`. Чи прочитав свій приймач повідомлення? У скільки разів довше тепер триває передача кожного біта?
4. У короткому звіті запишіть, що захищає повідомлення від чужого приймача, що дає довший код і чим за нього платять.